In [1]:
import pandas as pd
import json
import os

# 1. Create directory structure
os.makedirs('01_data/processed', exist_ok=True)
os.makedirs('04_python', exist_ok=True)

# 2. Load files
ledger = pd.read_csv('ledger.csv')
gateway = pd.read_csv('gateway.csv')

# 3. Check duplicates and nulls (outputs to console)
print(f"Ledger - Nulls: {ledger.isnull().sum().sum()}, Duplicates: {ledger.duplicated().sum()}")
print(f"Gateway - Nulls: {gateway.isnull().sum().sum()}, Duplicates: {gateway.duplicated().sum()}")

# 4. Reconciliation Logic using Outer Join
# Assuming 'transaction_id' is the common unique key
recon = pd.merge(
    ledger,
    gateway,
    on='transaction_id',
    how='outer',
    suffixes=('_ledger', '_gateway'),
    indicator=True
)

# 5. Identify specific discrepancies
missing_in_gateway = recon[recon['_merge'] == 'left_only']
missing_in_ledger = recon[recon['_merge'] == 'right_only']

# Filter for matched records to check data mismatches
matched = recon[recon['_merge'] == 'both']
amount_mismatches = matched[matched['amount_usd_ledger'] != matched['amount_usd_gateway']]
status_mismatches = matched[matched['status_ledger'] != matched['status_gateway']]

# 6. Save output files
missing_in_gateway.to_csv('01_data/processed/missing_in_gateway.csv', index=False)
missing_in_ledger.to_csv('01_data/processed/missing_in_ledger.csv', index=False)
amount_mismatches.to_csv('01_data/processed/amount_mismatches.csv', index=False)
status_mismatches.to_csv('01_data/processed/status_mismatches.csv', index=False)
recon.to_csv('01_data/processed/reconciliation_report.csv', index=False)

# 7. Generate Summary Metrics
metrics = {
    "total_ledger": len(ledger),
    "total_gateway": len(gateway),
    "missing_in_gateway": len(missing_in_gateway),
    "missing_in_ledger": len(missing_in_ledger),
    "amount_mismatches": len(amount_mismatches),
    "status_mismatches": len(status_mismatches)
}

with open('04_python/summary_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

print("Workflow complete. Check the 'Files' pane for results.")

Ledger - Nulls: 0, Duplicates: 0
Gateway - Nulls: 0, Duplicates: 0


KeyError: 'amount_ledger'

In [3]:
import pandas as pd
import json
import os

# 1. Create directory structure
os.makedirs('01_data/processed', exist_ok=True)
os.makedirs('04_python', exist_ok=True)

# 2. Load files
ledger = pd.read_csv('ledger.csv')
gateway = pd.read_csv('gateway.csv')

# 3. Check duplicates and nulls
print(f"Ledger - Nulls: {ledger.isnull().sum().sum()}, Duplicates: {ledger.duplicated().sum()}")
print(f"Gateway - Nulls: {gateway.isnull().sum().sum()}, Duplicates: {gateway.duplicated().sum()}")

# 4. Reconciliation Logic using Outer Join
recon = pd.merge(
    ledger,
    gateway,
    on='transaction_id',
    how='outer',
    suffixes=('_ledger', '_gateway'),
    indicator=True
)

# 5. Identify specific discrepancies
missing_in_gateway = recon[recon['_merge'] == 'left_only']
missing_in_ledger = recon[recon['_merge'] == 'right_only']

# Filter for matched records to check data mismatches
matched = recon[recon['_merge'] == 'both'].copy()

# Updated to use 'amount_usd' based on your file structure
amount_mismatches = matched[matched['amount_usd_ledger'] != matched['amount_usd_gateway']]
status_mismatches = matched[matched['status_ledger'] != matched['status_gateway']]

# 6. Save output files
missing_in_gateway.to_csv('01_data/processed/missing_in_gateway.csv', index=False)
missing_in_ledger.to_csv('01_data/processed/missing_in_ledger.csv', index=False)
amount_mismatches.to_csv('01_data/processed/amount_mismatches.csv', index=False)
status_mismatches.to_csv('01_data/processed/status_mismatches.csv', index=False)
recon.to_csv('01_data/processed/reconciliation_report.csv', index=False)

# 7. Generate Summary Metrics
metrics = {
    "total_ledger": len(ledger),
    "total_gateway": len(gateway),
    "missing_in_gateway": len(missing_in_gateway),
    "missing_in_ledger": len(missing_in_ledger),
    "amount_mismatches": len(amount_mismatches),
    "status_mismatches": len(status_mismatches)
}

with open('04_python/summary_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

print("--- Task Completed Successfully ---")
print("Files are now available in the '01_data/processed' and '04_python' folders.")

Ledger - Nulls: 0, Duplicates: 0
Gateway - Nulls: 0, Duplicates: 0
--- Task Completed Successfully ---
Files are now available in the '01_data/processed' and '04_python' folders.


In [4]:
import pandas as pd
import json

# 1. Read the nested JSON file
with open('api_response_sample.json', 'r') as f:
    api_data = json.load(f)

# 2. Flatten the data into tabular form
# We use json_normalize to break down nested dictionaries into columns
df_normalized = pd.json_normalize(api_data)

# 3. Clean column names
# Remove prefixes like 'payload.' or 'meta.' and replace dots with underscores
df_normalized.columns = [col.replace('.', '_').lower() for col in df_normalized.columns]

# 4. Convert date/time fields
# Identifying common timestamp columns (adjust names if your JSON keys differ)
date_cols = [col for col in df_normalized.columns if 'date' in col or 'time' in col]
for col in date_cols:
    df_normalized[col] = pd.to_datetime(df_normalized[col])

# 5. Save the normalized output
output_path = '01_data/processed/api_normalized.csv'
df_normalized.to_csv(output_path, index=False)

print(f"--- Part 4 Completed ---")
print(f"Normalized data saved to: {output_path}")
print(f"Columns created: {list(df_normalized.columns)}")

--- Part 4 Completed ---
Normalized data saved to: 01_data/processed/api_normalized.csv
Columns created: ['generated_at', 'source', 'batches']


In [5]:
import pandas as pd
import json
import os


os.makedirs('01_data/processed', exist_ok=True)
os.makedirs('04_python', exist_ok=True)

ledger = pd.read_csv('ledger.csv')
gateway = pd.read_csv('gateway.csv')


print(f"Ledger - Nulls: {ledger.isnull().sum().sum()}, Duplicates: {ledger.duplicated().sum()}")
print(f"Gateway - Nulls: {gateway.isnull().sum().sum()}, Duplicates: {gateway.duplicated().sum()}")


recon = pd.merge(
    ledger,
    gateway,
    on='transaction_id',
    how='outer',
    suffixes=('_ledger', '_gateway'),
    indicator=True
)

missing_in_gateway = recon[recon['_merge'] == 'left_only']
missing_in_ledger = recon[recon['_merge'] == 'right_only']


matched = recon[recon['_merge'] == 'both'].copy()


amount_mismatches = matched[matched['amount_usd_ledger'] != matched['amount_usd_gateway']]
status_mismatches = matched[matched['status_ledger'] != matched['status_gateway']]


missing_in_gateway.to_csv('01_data/processed/missing_in_gateway.csv', index=False)
missing_in_ledger.to_csv('01_data/processed/missing_in_ledger.csv', index=False)
amount_mismatches.to_csv('01_data/processed/amount_mismatches.csv', index=False)
status_mismatches.to_csv('01_data/processed/status_mismatches.csv', index=False)
recon.to_csv('01_data/processed/reconciliation_report.csv', index=False)

metrics = {
    "total_ledger": len(ledger),
    "total_gateway": len(gateway),
    "missing_in_gateway": len(missing_in_gateway),
    "missing_in_ledger": len(missing_in_ledger),
    "amount_mismatches": len(amount_mismatches),
    "status_mismatches": len(status_mismatches)
}

with open('04_python/summary_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

print("--- Task Completed Successfully ---")
print("Files are now available in the '01_data/processed' and '04_python' folders.")


import pandas as pd
import json


with open('api_response_sample.json', 'r') as f:
    api_data = json.load(f)



df_normalized = pd.json_normalize(api_data)



df_normalized.columns = [col.replace('.', '_').lower() for col in df_normalized.columns]



date_cols = [col for col in df_normalized.columns if 'date' in col or 'time' in col]
for col in date_cols:
    df_normalized[col] = pd.to_datetime(df_normalized[col])


output_path = '01_data/processed/api_normalized.csv'
df_normalized.to_csv(output_path, index=False)

print(f"--- Part 4 Completed ---")
print(f"Normalized data saved to: {output_path}")
print(f"Columns created: {list(df_normalized.columns)}")

Ledger - Nulls: 0, Duplicates: 0
Gateway - Nulls: 0, Duplicates: 0
--- Task Completed Successfully ---
Files are now available in the '01_data/processed' and '04_python' folders.
--- Part 4 Completed ---
Normalized data saved to: 01_data/processed/api_normalized.csv
Columns created: ['generated_at', 'source', 'batches']


In [1]:
import pandas as pd
import json
import os

os.makedirs('01_data/processed', exist_ok=True)
os.makedirs('04_python', exist_ok=True)

ledger = pd.read_csv('ledger.csv')
gateway = pd.read_csv('gateway.csv')

print(f"Ledger - Nulls: {ledger.isnull().sum().sum()}, Duplicates: {ledger.duplicated().sum()}")
print(f"Gateway - Nulls: {gateway.isnull().sum().sum()}, Duplicates: {gateway.duplicated().sum()}")

recon = pd.merge(
    ledger,
    gateway,
    on='transaction_id',
    how='outer',
    suffixes=('_ledger', '_gateway'),
    indicator=True
)

missing_in_gateway = recon[recon['_merge'] == 'left_only']
missing_in_ledger = recon[recon['_merge'] == 'right_only']
matched = recon[recon['_merge'] == 'both'].copy()

amount_mismatches = matched[matched['amount_usd_ledger'] != matched['amount_usd_gateway']]
status_mismatches = matched[matched['status_ledger'] != matched['status_gateway']]

missing_in_gateway.to_csv('01_data/processed/missing_in_gateway.csv', index=False)
missing_in_ledger.to_csv('01_data/processed/missing_in_ledger.csv', index=False)
amount_mismatches.to_csv('01_data/processed/amount_mismatches.csv', index=False)
status_mismatches.to_csv('01_data/processed/status_mismatches.csv', index=False)
recon.to_csv('01_data/processed/reconciliation_report.csv', index=False)

metrics = {
    "total_ledger": len(ledger),
    "total_gateway": len(gateway),
    "missing_in_gateway": len(missing_in_gateway),
    "missing_in_ledger": len(missing_in_ledger),
    "amount_mismatches": len(amount_mismatches),
    "status_mismatches": len(status_mismatches)
}

with open('04_python/summary_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

with open('api_response_sample.json', 'r') as f:
    api_data = json.load(f)

df_normalized = pd.json_normalize(api_data)
df_normalized.columns = [col.replace('.', '_').lower() for col in df_normalized.columns]

date_cols = [col for col in df_normalized.columns if 'date' in col or 'time' in col]
for col in date_cols:
    df_normalized[col] = pd.to_datetime(df_normalized[col])

df_normalized.to_csv('01_data/processed/api_normalized.csv', index=False)

print("--- All Tasks Completed Successfully ---")

FileNotFoundError: [Errno 2] No such file or directory: 'ledger.csv'

In [2]:
import pandas as pd
import json
import os


os.makedirs('01_data/processed', exist_ok=True)
os.makedirs('04_python', exist_ok=True)

ledger = pd.read_csv('ledger.csv')
gateway = pd.read_csv('gateway.csv')


print(f"Ledger - Nulls: {ledger.isnull().sum().sum()}, Duplicates: {ledger.duplicated().sum()}")
print(f"Gateway - Nulls: {gateway.isnull().sum().sum()}, Duplicates: {gateway.duplicated().sum()}")


recon = pd.merge(
    ledger,
    gateway,
    on='transaction_id',
    how='outer',
    suffixes=('_ledger', '_gateway'),
    indicator=True
)

missing_in_gateway = recon[recon['_merge'] == 'left_only']
missing_in_ledger = recon[recon['_merge'] == 'right_only']


matched = recon[recon['_merge'] == 'both'].copy()


amount_mismatches = matched[matched['amount_usd_ledger'] != matched['amount_usd_gateway']]
status_mismatches = matched[matched['status_ledger'] != matched['status_gateway']]


missing_in_gateway.to_csv('01_data/processed/missing_in_gateway.csv', index=False)
missing_in_ledger.to_csv('01_data/processed/missing_in_ledger.csv', index=False)
amount_mismatches.to_csv('01_data/processed/amount_mismatches.csv', index=False)
status_mismatches.to_csv('01_data/processed/status_mismatches.csv', index=False)
recon.to_csv('01_data/processed/reconciliation_report.csv', index=False)

metrics = {
    "total_ledger": len(ledger),
    "total_gateway": len(gateway),
    "missing_in_gateway": len(missing_in_gateway),
    "missing_in_ledger": len(missing_in_ledger),
    "amount_mismatches": len(amount_mismatches),
    "status_mismatches": len(status_mismatches)
}

with open('04_python/summary_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

print("--- Task Completed Successfully ---")
print("Files are now available in the '01_data/processed' and '04_python' folders.")


import pandas as pd
import json


with open('api_response_sample.json', 'r') as f:
    api_data = json.load(f)



df_normalized = pd.json_normalize(api_data)



df_normalized.columns = [col.replace('.', '_').lower() for col in df_normalized.columns]



date_cols = [col for col in df_normalized.columns if 'date' in col or 'time' in col]
for col in date_cols:
    df_normalized[col] = pd.to_datetime(df_normalized[col])


output_path = '01_data/processed/api_normalized.csv'
df_normalized.to_csv(output_path, index=False)

print(f"--- Part 4 Completed ---")
print(f"Normalized data saved to: {output_path}")
print(f"Columns created: {list(df_normalized.columns)}")

FileNotFoundError: [Errno 2] No such file or directory: 'ledger.csv'

In [3]:
import pandas as pd
import json
import os


os.makedirs('01_data/processed', exist_ok=True)
os.makedirs('04_python', exist_ok=True)

ledger = pd.read_csv('ledger.csv')
gateway = pd.read_csv('gateway.csv')


print(f"Ledger - Nulls: {ledger.isnull().sum().sum()}, Duplicates: {ledger.duplicated().sum()}")
print(f"Gateway - Nulls: {gateway.isnull().sum().sum()}, Duplicates: {gateway.duplicated().sum()}")


recon = pd.merge(
    ledger,
    gateway,
    on='transaction_id',
    how='outer',
    suffixes=('_ledger', '_gateway'),
    indicator=True
)

missing_in_gateway = recon[recon['_merge'] == 'left_only']
missing_in_ledger = recon[recon['_merge'] == 'right_only']


matched = recon[recon['_merge'] == 'both'].copy()


amount_mismatches = matched[matched['amount_usd_ledger'] != matched['amount_usd_gateway']]
status_mismatches = matched[matched['status_ledger'] != matched['status_gateway']]


missing_in_gateway.to_csv('01_data/processed/missing_in_gateway.csv', index=False)
missing_in_ledger.to_csv('01_data/processed/missing_in_ledger.csv', index=False)
amount_mismatches.to_csv('01_data/processed/amount_mismatches.csv', index=False)
status_mismatches.to_csv('01_data/processed/status_mismatches.csv', index=False)
recon.to_csv('01_data/processed/reconciliation_report.csv', index=False)

metrics = {
    "total_ledger": len(ledger),
    "total_gateway": len(gateway),
    "missing_in_gateway": len(missing_in_gateway),
    "missing_in_ledger": len(missing_in_ledger),
    "amount_mismatches": len(amount_mismatches),
    "status_mismatches": len(status_mismatches)
}

with open('04_python/summary_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)

print("--- Task Completed Successfully ---")
print("Files are now available in the '01_data/processed' and '04_python' folders.")


import pandas as pd
import json


with open('api_response_sample.json', 'r') as f:
    api_data = json.load(f)



df_normalized = pd.json_normalize(api_data)



df_normalized.columns = [col.replace('.', '_').lower() for col in df_normalized.columns]



date_cols = [col for col in df_normalized.columns if 'date' in col or 'time' in col]
for col in date_cols:
    df_normalized[col] = pd.to_datetime(df_normalized[col])


output_path = '01_data/processed/api_normalized.csv'
df_normalized.to_csv(output_path, index=False)

print(f"--- Part 4 Completed ---")
print(f"Normalized data saved to: {output_path}")
print(f"Columns created: {list(df_normalized.columns)}")

Ledger - Nulls: 0, Duplicates: 0
Gateway - Nulls: 0, Duplicates: 0
--- Task Completed Successfully ---
Files are now available in the '01_data/processed' and '04_python' folders.
--- Part 4 Completed ---
Normalized data saved to: 01_data/processed/api_normalized.csv
Columns created: ['generated_at', 'source', 'batches']
